# LLM Few-Shot (Mistral-7B) — Results

Evaluates document-level sentiment predictions from `llm_predictions.csv`.

- `llm_predictions.csv` already contains only the test set (24,849 rows)
- Ground truth column: `sentiment` (`positive` / `negative`)
- Prediction column: `pred_sentiment` (`positive` / `negative` / `N/A`)
- No train/test split needed — the file IS the test set

In [1]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [2]:
# Load predictions (already the test set)
df = pd.read_csv("llm_predictions.csv")
print(f"Total rows: {len(df)}")
print(f"\nsentiment distribution:\n{df['sentiment'].value_counts(dropna=False)}")
print(f"\npred_sentiment distribution:\n{df['pred_sentiment'].value_counts(dropna=False)}")

Total rows: 24849

sentiment distribution:
sentiment
positive    21144
negative     3705
Name: count, dtype: int64

pred_sentiment distribution:
pred_sentiment
positive    20265
negative     4430
NaN           154
Name: count, dtype: int64


In [3]:
# Drop N/A predictions
df_valid = df[df["pred_sentiment"].isin(["positive", "negative"])].copy()
n_na = len(df) - len(df_valid)
print(f"N/A predictions dropped: {n_na} ({n_na/len(df)*100:.1f}%)")
print(f"Evaluated on: {len(df_valid)} samples")

N/A predictions dropped: 154 (0.6%)
Evaluated on: 24695 samples


In [4]:
# Encode labels: positive=1, negative=0
y_true = (df_valid["sentiment"] == "positive").astype(int).values
y_pred = (df_valid["pred_sentiment"] == "positive").astype(int).values

print("=" * 60)
print("Overall Classification Report")
print("=" * 60)
print(classification_report(y_true, y_pred, target_names=["negative", "positive"]))

acc    = accuracy_score(y_true, y_pred)
mf1    = f1_score(y_true, y_pred, average="macro")
neg_f1 = f1_score(y_true, y_pred, pos_label=0)
pos_f1 = f1_score(y_true, y_pred, pos_label=1)
print(f"Accuracy : {acc:.4f}")
print(f"Macro F1 : {mf1:.4f}")
print(f"Neg F1   : {neg_f1:.4f}")
print(f"Pos F1   : {pos_f1:.4f}")

Overall Classification Report
              precision    recall  f1-score   support

    negative       0.80      0.96      0.87      3681
    positive       0.99      0.96      0.98     21014

    accuracy                           0.96     24695
   macro avg       0.90      0.96      0.92     24695
weighted avg       0.96      0.96      0.96     24695

Accuracy : 0.9585
Macro F1 : 0.9244
Neg F1   : 0.8736
Pos F1   : 0.9752


In [5]:
# Per-topic breakdown (valid predictions only)
df_topic = df_valid[df_valid["topic_label"].notna()]
topics = sorted(df_topic["topic_label"].unique())

print(f"{'Topic':<45} {'N':>6} {'Acc':>7} {'MacroF1':>9}")
print("-" * 70)

for topic in topics:
    subset = df_topic[df_topic["topic_label"] == topic]
    yt = (subset["sentiment"] == "positive").astype(int).values
    yp = (subset["pred_sentiment"] == "positive").astype(int).values
    n   = len(subset)
    acc = accuracy_score(yt, yp)
    mf1 = f1_score(yt, yp, average="macro", zero_division=0)
    print(f"{topic:<45} {n:>6,} {acc:>7.3f} {mf1:>9.3f}")

Topic                                              N     Acc   MacroF1
----------------------------------------------------------------------
Accessories                                    1,767   0.950     0.885
Acoustic tone                                    869   0.994     0.927
Beginner learning                              2,569   0.975     0.908
Customer service / returns                     1,618   0.918     0.917
Electronics / controls                           413   0.923     0.868
Fret / neck setup                              2,366   0.926     0.906
Guitar size                                    1,053   0.966     0.896
Pickups                                        1,474   0.967     0.773
Playability / chords                           1,077   0.966     0.893
Setup / action                                   980   0.984     0.903
Shipping damage                                1,112   0.913     0.912
String quality                                 1,062   0.910     0.907
Tuning